# Stream Aggregations, Windows and Watermarks

The pipeline in {doc}`Spark Structured Streaming <02_spark_structured_streaming>`
was **stateless**: each row was cleaned and enriched on its own, without
looking at any other row. The questions businesses ask of a stream are
usually about groups of rows, though: how many pastries were sold per
product, how much revenue each bakery made per hour. Answering them means
remembering what earlier micro-batches saw. This notebook covers these
**stateful** operations: aggregations, event-time windows, watermarks for
late data, and de-duplication.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_03")
spark.sql("CREATE VOLUME workspace.default.streaming_03")
for table in ["stream03_orders", "stream03_product_totals", "stream03_window_counts", "stream03_daily_revenue", "stream03_deduplicated"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")

CHECKPOINTS = "/Volumes/workspace/default/streaming_03/checkpoints"

## State

A streaming aggregation cannot recompute its result from the whole input on
every micro-batch, because the input never ends. Instead, Spark keeps the
running aggregate of every group in a **state store**, and each micro-batch
updates it with the new rows only.

```mermaid
:caption: Each micro-batch combines the new rows with the state left by the previous batch, writes the updated result and saves the new state.
:align: center

graph LR
    B1["Batch 1: new rows"] --> S1[("State after batch 1")]
    S1 --> B2["Batch 2: new rows + state"]
    B2 --> S2[("State after batch 2")]
    S2 --> B3["Batch 3: new rows + state"]
```

Structured Streaming checkpoints the state to fault-tolerant storage and
restores it after a restart
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).
Stateful operations include streaming aggregations, `dropDuplicates` and
stream-stream joins, and for managing their state Databricks recommends the
RocksDB state store with changelog checkpointing
([What is stateful streaming?](https://docs.databricks.com/aws/en/structured-streaming/stateful-streaming#optimize-stateful-structured-streaming-queries)).
State is never free, though: it grows with every new group, and keeping it
bounded is the main design concern of stateful streaming.

## A Stream to Work With

We need a source that receives new data while the notebook runs. A small
Delta table of bakery orders plays the part of the bronze table from
{doc}`Spark Structured Streaming <02_spark_structured_streaming>`: each call
to `land` appends a few orders to it, the way an ingestion stream would.

In [2]:
spark.sql("""
    CREATE TABLE workspace.default.stream03_orders (
        event_time TIMESTAMP, product STRING, quantity INT
    )
""")


def land(*orders: tuple[str, str, int]) -> None:
    """Append orders, given as (event time, product, quantity), to the source table."""
    (
        spark.createDataFrame(orders, "event_time STRING, product STRING, quantity INT")
        .withColumn("event_time", F.to_timestamp("event_time"))
        .write.mode("append")
        .saveAsTable("workspace.default.stream03_orders")
    )


orders = spark.readStream.table("workspace.default.stream03_orders")

## Aggregations without Windows

Grouping a stream works just like grouping a static DataFrame. This query
keeps a running total per product:

In [3]:
product_totals = orders.groupBy("product").agg(
    F.count("*").alias("orders"),
    F.sum("quantity").alias("items"),
)


def run_product_totals():
    query = (
        product_totals.writeStream
        .outputMode("complete")
        .option("checkpointLocation", f"{CHECKPOINTS}/product_totals")
        .trigger(availableNow=True)
        .toTable("workspace.default.stream03_product_totals")
    )
    query.awaitTermination()
    return spark.read.table("workspace.default.stream03_product_totals").orderBy("product")

`complete` mode emits all result rows on each trigger
([Select an output mode for Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/output-mode#available-output-modes)),
which is fine for a handful of products. Land three orders and run the query:

In [4]:
land(
    ("2024-05-01 10:02:00", "Pearly Pies", 2),
    ("2024-05-01 10:06:00", "Tokyo Tidbits", 1),
    ("2024-05-01 10:11:00", "Pearly Pies", 3),
)

run_product_totals().show()

+-------------+------+-----+
|      product|orders|items|
+-------------+------+-----+
|  Pearly Pies|     2|    5|
|Tokyo Tidbits|     1|    1|
+-------------+------+-----+



Two more orders arrive. The second run reads only these two rows, and
combines them with the totals kept in state:

In [5]:
land(
    ("2024-05-01 10:14:00", "Pearly Pies", 1),
    ("2024-05-01 10:23:00", "Orchard Oasis", 4),
)

run_product_totals().show()

+-------------+------+-----+
|      product|orders|items|
+-------------+------+-----+
|Orchard Oasis|     1|    4|
|  Pearly Pies|     3|    6|
|Tokyo Tidbits|     1|    1|
+-------------+------+-----+



Pearly Pies now has 3 orders and $2 + 3 + 1 = 6$ items, although the second
run never re-read the first three orders.

This query has a problem in a real stream, though. Its state holds one row
per product *forever*. That is harmless for six products, but grouping by
customer, or by anything with a growing number of values, makes the state
grow without limit. Time-based questions solve this with **windows** and
**watermarks**.

## Event-Time Windows

Most streaming aggregations are about time periods: orders per hour,
revenue per day. `F.window(time_column, duration)` assigns each row to a
time window based on its **event time**, as discussed in
{doc}`Introduction to Stream Processing <01_introduction_to_stream_processing>`,
and is used as a grouping column. Spark supports three kinds of windows
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)):

| Window | Created with | Each event belongs to |
| --- | --- | --- |
| Tumbling | `F.window("event_time", "10 minutes")` | Exactly one fixed, non-overlapping window |
| Sliding | `F.window("event_time", "10 minutes", "5 minutes")` | Every window it falls into; windows overlap |
| Session | `F.session_window("event_time", "5 minutes")` | A window that grows while events keep arriving, and closes after 5 minutes of inactivity |

Windows start at `1970-01-01 00:00:00 UTC` plus an optional offset
([window grouping expression](https://docs.databricks.com/aws/en/sql/language-manual/functions/window)),
so 10-minute windows always start at :00, :10, :20 and so on. A window
includes its start and excludes its end, as the check below shows for an
order at exactly 10:10.

:::::{dropdown} Example: Which Windows Contain an Event?
:open:
:color: primary
:icon: pencil
:class-body: example-body

An order is placed at 10:07. Which windows is it counted in?

**Tumbling, 10 minutes.** The windows are 10:00–10:10, 10:10–10:20, and so
on. 10:07 falls into exactly one of them:

```text
[10:00, 10:10)
```

**Sliding, 10 minutes long, every 5 minutes.** A new 10-minute window starts
every 5 minutes, so each moment is covered by $10 / 5 = 2$ windows:

```text
[09:55, 10:05)   no:  10:07 is after the end
[10:00, 10:10)   yes
[10:05, 10:15)   yes
[10:10, 10:20)   no:  10:07 is before the start
```

The order is counted in two sliding windows. Sliding windows are useful for
smoothed metrics such as "orders in the last 10 minutes, updated every 5",
but they multiply both the output and the state by the
length-to-slide ratio.
:::::

Checking the example with a static DataFrame, since `window` works the same
way in batch queries:

In [6]:
two_orders = spark.createDataFrame([("2024-05-01 10:07:00",), ("2024-05-01 10:10:00",)], "event_time STRING").select(
    F.to_timestamp("event_time").alias("event_time")
)

two_orders.select("event_time", F.window("event_time", "10 minutes").alias("tumbling")).show(truncate=False)
two_orders.filter(F.minute("event_time") == 7).select(
    "event_time", F.window("event_time", "10 minutes", "5 minutes").alias("sliding")
).show(truncate=False)

+-------------------+------------------------------------------+
|event_time         |tumbling                                  |
+-------------------+------------------------------------------+
|2024-05-01 10:07:00|{2024-05-01 10:00:00, 2024-05-01 10:10:00}|
|2024-05-01 10:10:00|{2024-05-01 10:10:00, 2024-05-01 10:20:00}|
+-------------------+------------------------------------------+



+-------------------+------------------------------------------+
|event_time         |sliding                                   |
+-------------------+------------------------------------------+
|2024-05-01 10:07:00|{2024-05-01 10:05:00, 2024-05-01 10:15:00}|
|2024-05-01 10:07:00|{2024-05-01 10:00:00, 2024-05-01 10:10:00}|
+-------------------+------------------------------------------+



The 10:07 order lands in `[10:00, 10:10)` and in two sliding windows; the
10:10 order already belongs to the next tumbling window, `[10:10, 10:20)`.

## Watermarks: Deciding When a Window Is Final

A window aggregation still has the state problem: since a late order for
10:00–10:10 could in principle arrive at any time, Spark would have to keep
every window in state forever. And in `append` mode it could never write a
window out, because a window might still change.

A **watermark** is how you tell Spark how late data can be. With
`withWatermark("event_time", "10 minutes")`, Spark tracks the latest event
time it has seen and computes
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html))

$$
\text{watermark} = \max(\text{event time seen so far}) - 10 \text{ minutes}.
$$ (eq-watermark)

The watermark is a promise that you do not expect events older than it any
more. That has three consequences
([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks)):

- A window whose **end** is at or before the watermark is **final**. In
  `append` mode, it is written to the sink at this point, once.
- Its old aggregation state is then **dropped** from the state store, so
  state stays bounded.
- Rows that arrive later than the threshold are no longer guaranteed to be
  processed. Data within the threshold is always aggregated; data beyond it
  may or may not be
  ([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).
  In the runs below, the late rows are dropped.

The watermark is computed from the data of one trigger and applied in the
next one
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).
It only ever moves forward: an old event never pulls it back.

Here is a windowed count with a 10-minute watermark, written in `append`
mode. The orders landed above are already in the source table, and the
query starts from the beginning of it:

In [7]:
window_counts = (
    orders
    .withWatermark("event_time", "10 minutes")
    .groupBy(F.window("event_time", "10 minutes"))
    .agg(F.count("*").alias("orders"), F.sum("quantity").alias("items"))
)


def run_window_counts():
    query = (
        window_counts.writeStream
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINTS}/window_counts")
        .trigger(availableNow=True)
        .toTable("workspace.default.stream03_window_counts")
    )
    query.awaitTermination()
    for progress in query.recentProgress:
        state = progress["stateOperators"][0]
        print(
            f"batch {progress['batchId']}: "
            f"read {progress['sources'][0]['numInputRows']} rows, "
            f"watermark {progress['eventTime'].get('watermark')}, "
            f"windows in state {state['numRowsTotal']}, "
            f"late rows dropped {state['numRowsDroppedByWatermark']}"
        )
    return spark.read.table("workspace.default.stream03_window_counts").orderBy("window")

In [8]:
run_window_counts().show(truncate=False)

batch 0: read 5 rows, watermark 1970-01-01T00:00:00.000Z, windows in state 3, late rows dropped 0
batch 1: read 0 rows, watermark 2024-05-01T10:13:00.000Z, windows in state 2, late rows dropped 0


+------------------------------------------+------+-----+
|window                                    |orders|items|
+------------------------------------------+------+-----+
|{2024-05-01 10:00:00, 2024-05-01 10:10:00}|2     |3    |
+------------------------------------------+------+-----+



Watch this run, and the next two, closely. The progress lines show each
micro-batch in turn. When a run moves the watermark forward, Spark adds a
batch that reads no rows, only to apply the new watermark. The dropdown
below walks through all three runs.

The next delivery contains a late order: one for 10:04, which arrives after
orders for 10:23 have already been seen.

In [9]:
land(
    ("2024-05-01 10:31:00", "Pearly Pies", 2),
    ("2024-05-01 10:04:00", "Golden Gate Ginger", 2),
)

run_window_counts().show(truncate=False)

batch 2: read 2 rows, watermark 2024-05-01T10:13:00.000Z, windows in state 3, late rows dropped 1
batch 3: read 0 rows, watermark 2024-05-01T10:21:00.000Z, windows in state 2, late rows dropped 0


+------------------------------------------+------+-----+
|window                                    |orders|items|
+------------------------------------------+------+-----+
|{2024-05-01 10:00:00, 2024-05-01 10:10:00}|2     |3    |
|{2024-05-01 10:10:00, 2024-05-01 10:20:00}|2     |4    |
+------------------------------------------+------+-----+



And a third delivery, with an order for 10:42 and an order that is *very*
late, for 10:07:

In [10]:
land(
    ("2024-05-01 10:42:00", "Outback Oatmeal", 1),
    ("2024-05-01 10:07:00", "Tokyo Tidbits", 5),
)

run_window_counts().show(truncate=False)

batch 4: read 2 rows, watermark 2024-05-01T10:21:00.000Z, windows in state 3, late rows dropped 1
batch 5: read 0 rows, watermark 2024-05-01T10:32:00.000Z, windows in state 2, late rows dropped 0


+------------------------------------------+------+-----+
|window                                    |orders|items|
+------------------------------------------+------+-----+
|{2024-05-01 10:00:00, 2024-05-01 10:10:00}|2     |3    |
|{2024-05-01 10:10:00, 2024-05-01 10:20:00}|2     |4    |
|{2024-05-01 10:20:00, 2024-05-01 10:30:00}|1     |4    |
+------------------------------------------+------+-----+



:::::{dropdown} Example: Following the Watermark Through Three Runs
:open:
:color: primary
:icon: pencil
:class-body: example-body

The watermark delay is 10 minutes and the windows are 10 minutes long. All
times are on 1 May 2024.

**Run 1** reads the five orders landed earlier, at 10:02, 10:06, 10:11,
10:14 and 10:23. The watermark starts at the beginning of time, so nothing
is late. Afterwards, by {eq}`eq-watermark`,

$$
\text{watermark} = 10{:}23 - 10 \text{ min} = 10{:}13.
$$

The 10:00–10:10 window ends at 10:10, which is before 10:13, so it is final
and written with its 2 orders (10:02 and 10:06). The windows 10:10–10:20 and
10:20–10:30 are still open.

**Run 2** reads the orders for 10:31 and 10:04, with the watermark still at
10:13. The 10:04 order belongs to 10:00–10:10, which ended before the
watermark, so it is **dropped**: that window has been written already and
will not change. The 10:31 order opens the 10:30–10:40 window. Afterwards,

$$
\text{watermark} = 10{:}31 - 10 \text{ min} = 10{:}21,
$$

which finalises 10:10–10:20 with its 2 orders (10:11 and 10:14).

**Run 3** reads the orders for 10:42 and 10:07. The 10:07 order is dropped
for the same reason as the 10:04 order. Afterwards,

$$
\text{watermark} = 10{:}42 - 10 \text{ min} = 10{:}32,
$$

which finalises 10:20–10:30 with its single order (10:23). The 10:30–10:40
and 10:40–10:50 windows stay open until the watermark passes their ends.

Each window was written exactly once, after it was final, and two late
orders were dropped. The progress lines above report the same watermarks
and the same dropped rows.
:::::

Choosing the watermark delay is a trade-off. A longer delay has a high
tolerance for late data, but increases query latency and keeps more state. A
shorter delay lowers latency, but has a low tolerance for late data
([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#how-do-watermarks-affect-processing-time-and-throughput)). Pick it from
how late your data actually arrives, which you can measure by comparing
event time with ingestion time.

:::{admonition} Note: Watermarks and Output Modes
:class: note

In `append` mode, a window is written once, after the watermark finalises
it. In `update` mode, a window is written on every trigger that changes it,
so results appear sooner
([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#watermarks-and-output-mode-for-windowed-aggregations)),
but the sink must be able to overwrite earlier versions of a row. A Delta
sink supports `append` and `complete` but not `update`
([Select an output mode for Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/output-mode#operator-and-sink-compatibility)),
so `update` into Delta goes through `MERGE` in `foreachBatch`. The watermark
still removes old state in `update` mode. In `complete` mode, old state is
never dropped, because the whole result table is written every time
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).
:::

## Windowed Aggregations on Real Data

The same pattern applied to all bakehouse transactions gives revenue per
product per day. Here the source is the static sample table, read as a
stream:

In [11]:
daily_revenue = (
    spark.readStream.table("samples.bakehouse.sales_transactions")
    .withWatermark("dateTime", "1 hour")
    .groupBy(F.window("dateTime", "1 day").alias("day"), "product")
    .agg(F.sum("totalPrice").alias("revenue"), F.count("*").alias("transactions"))
)

query = (
    daily_revenue.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINTS}/daily_revenue")
    .trigger(availableNow=True)
    .toTable("workspace.default.stream03_daily_revenue")
)
query.awaitTermination()

result = spark.read.table("workspace.default.stream03_daily_revenue")
result.count()

96

The sales span 17 days and 6 products, so there are $17 \times 6 = 102$
windows, but only 96 were written. The latest sale in the table was at
11:48 on 17 May, so the watermark ended at 10:48 that day. The six windows
for 17 May end at midnight and are not final yet. They would be written by
a later run, once sales from 18 May push the watermark past midnight. The
latest window written is the one for 16 May, ending at midnight on 17 May:

In [12]:
result.select(F.max("day.end").alias("latest_window_written")).show()

+---------------------+
|latest_window_written|
+---------------------+
|  2024-05-17 00:00:00|
+---------------------+



## De-duplication

Sources often deliver the same event more than once, for example when an
upstream system retries a send. On a stream, `dropDuplicates(["id"])` must
remember every ID it has ever seen, so its state grows forever.
`dropDuplicatesWithinWatermark` bounds it: it requires a watermark, and it
always de-duplicates records that arrive within the watermark threshold
([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#drop-duplicates-within-watermark)).

In [13]:
deduplicated = (
    spark.readStream.table("samples.bakehouse.sales_transactions")
    .unionByName(spark.readStream.table("samples.bakehouse.sales_transactions"))
    .withWatermark("dateTime", "1 hour")
    .dropDuplicatesWithinWatermark(["transactionID"])
)

query = (
    deduplicated.writeStream
    .option("checkpointLocation", f"{CHECKPOINTS}/deduplicated")
    .trigger(availableNow=True)
    .toTable("workspace.default.stream03_deduplicated")
)
query.awaitTermination()

spark.read.table("workspace.default.stream03_deduplicated").count()

3333

The stream read every transaction twice, from a union of the table with
itself, and wrote each of the 3,333 transactions once. A duplicate is
caught as long as the second copy arrives within the watermark delay of the
first.

## Stream-Stream Joins

Two streams can be joined, for example orders with their deliveries. Each
side has to keep its rows in state until a matching row can no longer
arrive. To avoid unbounded state, define watermarks on both inputs and an
event-time constraint, such as a time range in the join condition, that
bounds how far apart matching events can be
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).
For inner joins Databricks recommends a watermark on each source; for outer
joins it is mandatory, and stream-stream joins support only `append` mode
([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#watermarks-and-output-modes-for-stream-stream-joins)):

```python
orders = spark.readStream.table("orders").withWatermark("order_time", "1 hour")
deliveries = spark.readStream.table("deliveries").withWatermark("delivery_time", "1 hour")

orders_with_delivery = orders.join(
    deliveries,
    F.expr("""
        orders.order_id = deliveries.order_id
        AND delivery_time BETWEEN order_time AND order_time + INTERVAL 2 HOURS
    """),
    "left",
)
```

Without the time condition, every order would have to be kept forever in
case a delivery for it turns up. For outer joins, the watermark and time
condition are also what let Spark decide that an order will never be
delivered and emit it with `NULL` delivery columns, after a delay that
depends on both
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)).

## Choosing a Stateful Pattern

| Question | Pattern | Output mode |
| --- | --- | --- |
| A running total per key with few keys | Aggregation without a window | `complete` |
| Totals per time period | Tumbling window plus watermark | `append` into Delta |
| A smoothed metric, such as the last 10 minutes every 5 | Sliding window plus watermark | `append` |
| Activity bursts per user | Session window plus watermark | `append` |
| Repeated deliveries of the same event | `dropDuplicatesWithinWatermark` | `append` |
| Matching events from two streams | Stream-stream join with watermarks and a time condition | `append` |

Databricks recommends `append` in most cases, `update` for sinks that need
the freshest results, and materialized views when you need complete-mode
semantics with incremental processing
([Select an output mode for Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/output-mode#application-semantics)).

## Common Mistakes

- **Grouping by an ever-growing key without a watermark.** State grows
  without limit, as the product totals example explained. Use a window and
  a watermark.
- **Writing a windowed aggregation in `append` mode and expecting instant
  results.** A window is written only after the watermark finalises it, so
  the watermark delay sets a minimum latency
  ([Select an output mode for Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/output-mode#latency-and-cost)).
- **Choosing `update` mode for a Delta sink.** Delta supports `append` and
  `complete` only; use `foreachBatch` with `MERGE`
  ([Select an output mode for Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/output-mode#operator-and-sink-compatibility)).
- **Picking a watermark delay without measuring lateness.** Too short drops
  late data, too long delays results and keeps more state
  ([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#how-do-watermarks-affect-processing-time-and-throughput)).
- **De-duplicating a stream with plain `dropDuplicates`.** Without a
  watermark it remembers every key forever. Use
  `dropDuplicatesWithinWatermark`.
- **A stream-stream outer join without watermarks.** Watermarks are
  mandatory for outer joins
  ([Apply watermarks to control data processing thresholds](https://docs.databricks.com/aws/en/structured-streaming/watermarks#watermarks-and-output-modes-for-stream-stream-joins)).

## Cleaning Up

In [14]:
for table in ["stream03_orders", "stream03_product_totals", "stream03_window_counts", "stream03_daily_revenue", "stream03_deduplicated"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")
spark.sql("DROP VOLUME IF EXISTS workspace.default.streaming_03");

Next, {doc}`Real-Time Mode in Structured Streaming <04_real_time_mode_in_structured_streaming>`
looks at what to do when micro-batch latency is not low enough.

## Resources

- [Apply watermarks to control data processing thresholds (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/watermarks)
- [What is stateful streaming? (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/stateful-streaming)
- [Select an output mode for Structured Streaming (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/output-mode)
- [Structured Streaming Programming Guide (Apache Spark)](https://spark.apache.org/docs/latest/streaming/index.html)